Copyright 2026, [AGH University of Krakow](https://www.agh.edu.pl/en) , [Institute of Telecommunications](https://tele.agh.edu.pl/)
Author: **Jaroslaw Bulat** kwant@agh.edu.pl, [LinkedIn](https://www.linkedin.com/in/jaros%C5%82aw-bu%C5%82at-30a9b191/), [WWW](https://home.agh.edu.pl/kwant/)

# **Yin Yang - Full Autoencoder**

Train a full-size MLP autoencoder on the Yin Yang dataset.

In [ ]:
# fetch repository when running on Colab
!git clone https://github.com/kwanty/YinYang.git YinYang-repo
%cd YinYang-repo

import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from yinyang import generate_dataset

## Load Dataset

In [ ]:
# Option 1: Generate a fresh dataset and save it
x_train, y_train = generate_dataset(10000)
# np.savez_compressed('../data/yinyang_10k.npz', x_train=x_train, y_train=y_train)

# Option 2: Load existing dataset from file
# data = np.load('../data/yinyang_10k.npz')
# x_train = data['x_train']
# y_train = data['y_train']

print(f'Dataset shape: {x_train.shape}')
print(f'Labels shape: {y_train.shape}')

## Build Full Autoencoder

Architecture:
- Encoder: 28 × 28 → dense layers → latent space
- Decoder: latent space → dense layers → 28 × 28

In [ ]:
# Define image dimensions parametrically
img_size = (28, 28)
latent_dim = 16

# Input layer
input_img = keras.layers.Input(shape=img_size)

# Encoder: flatten and compress to latent space
encoder = keras.Sequential([
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(latent_dim, activation='relu')
])

# Decoder: expand from latent space and reshape
decoder = keras.Sequential([
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(int(np.prod(img_size)), activation='sigmoid'),
    keras.layers.Reshape(img_size)
])

# Full autoencoder
autoencoder = decoder(encoder(input_img))

model = keras.Model(input_img, autoencoder)
print(model.summary(expand_nested=True))

model.compile(optimizer='adam', loss='mse', metrics=['MAE'])

## Train

In [ ]:
# Train the model
history = model.fit(
    x_train, x_train,
    epochs=30,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

## Evaluate and Visualize

In [ ]:
# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

## Reconstruction Examples

In [ ]:
# Get reconstructions
reconstructed = model.predict(x_train[:10])

# Visualize original vs reconstructed
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_train[i], cmap='gray')
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')

    axes[1, i].imshow(reconstructed[i], cmap='gray')
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')

plt.suptitle('Original vs Reconstructed (Full AE)', fontsize=12)
plt.tight_layout()
plt.show()

## Conclusions

The model reconstructs images quite well even with a latent space as small as 2  
(a vector of size 2 literally two numbers).

This means the model can compress an entire 28 × 28 image (784 numbers) into a  
single 2‑D vector (2 numbers).

A few experiments. You can improve the results even further, but there are  
better architectures for this task than a plain MLP.

- 1 + 1 hidden layer (encoder + decoder), latent space: 16 → MAE 0.0033  
- 1 + 1 hidden layer (encoder + decoder), latent space: 2  → MAE 0.0033  
- 2 + 2 hidden layer (encoder + decoder), latent space: 2  → MAE 0.0067

About MAE: the images are normalized to 0–1, so the smallest possible error per  
pixel is 1/256 = **0.00390625**.  
The model achieves reconstruction at the level of image‑quantization error!
